In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
YEAR_TO_ANALYZE = 2025


In [ ]:
# Load Dataset and Filter 2025 Records
file_path = Path('accidents_clean.csv')

if not file_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
    except ImportError:
        raise FileNotFoundError('Please place accidents_clean.csv in the project folder.')

if not file_path.exists():
    raise FileNotFoundError('accidents_clean.csv was not uploaded.')

df = pd.read_csv(file_path)
print('Original dataset shape:', df.shape)

date_columns = [col for col in df.columns if 'date' in col.lower()]
year_col = next((col for col in df.columns if col.lower() == 'year'), None)

if date_columns:
    date_col = date_columns[0]
    parsed_dates = pd.to_datetime(df[date_col], errors='coerce')
    if parsed_dates.notna().any():
        df[date_col] = parsed_dates
        df = df[df[date_col].dt.year == YEAR_TO_ANALYZE].copy()
    elif year_col is not None:
        df = df[df[year_col].astype(str) == str(YEAR_TO_ANALYZE)].copy()
    else:
        raise ValueError('Could not identify valid dates to filter 2025.')
elif year_col is not None:
    df = df[df[year_col].astype(str) == str(YEAR_TO_ANALYZE)].copy()
else:
    raise ValueError('No date or year column found. Cannot confirm 2025 records.')

if df.empty:
    raise ValueError('No records found for 2025 in accidents_clean.csv.')

print('2025 dataset loaded successfully!')
print('Dataset shape:', df.shape)
display(df.head())


In [ ]:
# Dataset Information
print('Dataset Information:')
df.info()
print('\nMissing Values:')
print(df.isnull().sum().sort_values(ascending=False).head(15))
print('\nDuplicate Rows:', df.duplicated().sum())


In [ ]:
# Identify Target Column
possible_targets = ['target', 'target_severe', 'severity', 'accident_severity', 'severity_level']
target_col = next((c for c in df.columns if c.lower() in possible_targets), None)

if target_col is None:
    target_col = next((c for c in df.columns if 'severity' in c.lower()), None)

if target_col is None:
    raise ValueError('Target column not found. Check your dataset columns.')

print('Target column:', target_col)
print(df[target_col].value_counts(dropna=False))


In [ ]:
# Prepare Binary Target
y_raw = df[target_col]

if pd.api.types.is_numeric_dtype(y_raw):
    unique_values = sorted(y_raw.dropna().unique())
    if set(unique_values).issubset({0, 1}):
        y = y_raw.copy()
    elif set(unique_values).issubset({1, 2, 3}):
        y = y_raw.map({1: 1, 2: 1, 3: 0})
    else:
        y = (y_raw == y_raw.max()).astype(int)
else:
    y_text = y_raw.astype(str).str.lower().str.strip()
    severe_words = ['fatal', 'serious', 'severe', 'high', 'major']
    y = y_text.apply(lambda value: int(any(word in value for word in severe_words)))

valid_rows = y.notna()
df = df.loc[valid_rows].copy()
y = y.loc[valid_rows].astype(int)

if y.nunique() < 2:
    raise ValueError('The target has only one class. A classification model needs at least two classes.')

print('Target distribution:')
print(y.value_counts())


In [ ]:
# Prepare Features
drop_columns = [target_col]

for col in df.columns:
    col_lower = col.lower()
    if col_lower in ['number_of_casualties', 'casualties', 'casualty_count']:
        drop_columns.append(col)
    if col_lower in ['id', 'accident_id', 'record_id']:
        drop_columns.append(col)

X = df.drop(columns=list(set(drop_columns)), errors='ignore')
X = X.dropna(axis=1, how='all')

for col in list(X.columns):
    if 'date' in col.lower():
        parsed_date = pd.to_datetime(X[col], errors='coerce')
        if parsed_date.notna().sum() > 0:
            X[col + '_year'] = parsed_date.dt.year
            X[col + '_month'] = parsed_date.dt.month
            X[col + '_day'] = parsed_date.dt.day
            X = X.drop(columns=[col])

print('Features shape:', X.shape)
display(X.head())


In [ ]:
# Identify Numerical and Categorical Features
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print('Numerical features:', len(numeric_features))
print('Categorical features:', len(categorical_features))


In [ ]:
# Split Dataset into Training, Validation, and Testing Sets
if y.value_counts().min() < 3:
    raise ValueError('Each target class needs at least 3 records for this split.')

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25,
    random_state=RANDOM_STATE, stratify=y_train_full
)

print('Training samples:', len(X_train))
print('Validation samples:', len(X_val))
print('Testing samples:', len(X_test))


In [ ]:
# Tune Decision Tree Maximum Depth
depth_values = [2, 3, 4, 5, 6, 8, 10, 12, 15, 20]
train_f1_scores = []
val_f1_scores = []

for depth in depth_values:
    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE, class_weight='balanced'))
    ])
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    train_f1_scores.append(f1_score(y_train, train_pred, zero_division=0))
    val_f1_scores.append(f1_score(y_val, val_pred, zero_division=0))

best_depth = depth_values[int(np.argmax(val_f1_scores))]
print('Best max_depth:', best_depth)

plt.figure(figsize=(9, 5))
plt.plot(depth_values, train_f1_scores, marker='o', label='Training F1')
plt.plot(depth_values, val_f1_scores, marker='o', label='Validation F1')
plt.xlabel('Maximum Tree Depth')
plt.ylabel('F1 Score')
plt.title('Decision Tree Depth Tuning - 2025')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Train Final Decision Tree Model
final_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        max_depth=best_depth,
        random_state=RANDOM_STATE,
        class_weight='balanced'
    ))
])

final_model.fit(X_train_full, y_train_full)
y_pred = final_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print('Decision Tree Model Results - 2025')
print('Accuracy:', round(accuracy, 4))
print('Precision:', round(precision, 4))
print('Recall:', round(recall, 4))
print('F1 Score:', round(f1, 4))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Severe', 'Severe'],
            yticklabels=['Not Severe', 'Severe'])
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Decision Tree Confusion Matrix - 2025')
plt.show()


In [ ]:
# Feature Importance
trained_preprocessor = final_model.named_steps['preprocessor']
trained_classifier = final_model.named_steps['classifier']
feature_names = trained_preprocessor.get_feature_names_out()
importances = trained_classifier.feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print('Top 10 Important Features:')
display(importance_df.head(10))

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(10), x='Importance', y='Feature')
plt.title('Top 10 Decision Tree Feature Importances - 2025')
plt.tight_layout()
plt.show()


In [ ]:
# Compare Maximum Depth Values
comparison_df = pd.DataFrame({
    'Max Depth': depth_values,
    'Training F1': train_f1_scores,
    'Validation F1': val_f1_scores
})
display(comparison_df)

plt.figure(figsize=(9, 5))
sns.barplot(data=comparison_df, x='Max Depth', y='Validation F1')
plt.title('Validation F1 Score Comparison - 2025')
plt.show()


In [ ]:
# Visualize Decision Tree
plt.figure(figsize=(20, 10))
plot_tree(
    trained_classifier,
    max_depth=3,
    feature_names=feature_names,
    class_names=['Not Severe', 'Severe'],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title('Decision Tree Visualization - 2025')
plt.show()


In [ ]:
# Save Predictions and Feature Importance
predictions_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred
})
predictions_df.to_csv('decision_tree_predictions_2025.csv', index=False)
importance_df.to_csv('decision_tree_feature_importance_2025.csv', index=False)
print('Output files saved successfully.')


In [ ]:
# Final Project Summary
print('=' * 55)
print('TRAFFIC ACCIDENT PATTERNS - DECISION TREE MODEL')
print('YEAR:', YEAR_TO_ANALYZE)
print('=' * 55)
print('Dataset:', file_path.name)
print('Number of records:', len(df))
print('Number of features:', X.shape[1])
print('Best max_depth:', best_depth)
print('Accuracy:', round(accuracy, 4))
print('Precision:', round(precision, 4))
print('Recall:', round(recall, 4))
print('F1 Score:', round(f1, 4))
print('=' * 55)
